# Shipping it, and reading the literature

> Why accuracy is usually the wrong number, what a threshold really is, how a model decays in production, and how to get the gist of a paper in fifteen minutes.

Read this chapter at `/learn/16-shipping-and-reading-papers/`. Exported from `src/content/chapters/16-shipping-and-reading-papers.mdx` — edit there, not here.


Last day. Two things separate a model from a system: knowing what number to
report, and being able to keep learning without anybody writing you a curriculum.

Let's do both, and then you're done.

## Accuracy is usually the wrong number

In [ ]:
import numpy as np, matplotlib.pyplot as plt

n = 10_000
truth = np.zeros(n, dtype=int)
truth[:100] = 1                      # 1% fraud
rng = np.random.default_rng(0)
rng.shuffle(truth)

always_no = np.zeros(n, dtype=int)
print(f"model that always says 'not fraud': accuracy {(always_no == truth).mean():.2%}")
print(f"frauds caught: {((always_no == 1) & (truth == 1)).sum()} of {truth.sum()}")

Ninety-nine percent accuracy. Zero frauds caught. Zero value.

You met this in chapter 3 and here it is again, because it's the mistake that
keeps happening. **Any metric that a constant can score well on is not measuring
your problem.**

In [ ]:
scores = np.where(truth == 1,
                  rng.normal(0.65, 0.20, n),
                  rng.normal(0.30, 0.18, n)).clip(0, 1)

def confusion(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    return tp, fp, fn, tn

tp, fp, fn, tn = confusion(truth, (scores > 0.5).astype(int))
print(f"                  predicted fraud   predicted ok")
print(f"actually fraud  {tp:14,d} {fn:14,d}")
print(f"actually ok     {fp:14,d} {tn:14,d}")
print(f"\nprecision {tp/(tp+fp):.3f}  — of those flagged, how many really were fraud")
print(f"recall    {tp/(tp+fn):.3f}  — of the real frauds, how many did we catch")

Four numbers instead of one. That's the real object, and accuracy is just a lossy
summary of it.

Learn these two by their **sentences**, not their formulas — formulas you can look
up, sentences you can use in a meeting:

- **Precision** — when the model says yes, how often is it right? Low precision
  means you're wasting effort chasing false alarms.
- **Recall** — of the things that really were true, how many did you find? Low
  recall means you missed them.

And they trade off against each other, necessarily. A model that flags
*everything* has recall 1.0 and dreadful precision. A model that flags only its
single most confident case has precision near 1.0 and useless recall.

Neither of those is a better model. They're the same model at different settings.

## The threshold is a business decision

In [ ]:
rows = []
for t in [0.2, 0.35, 0.5, 0.65, 0.8]:
    tp, fp, fn, tn = confusion(truth, (scores > t).astype(int))
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    rows.append((t, tp, fp, fn, prec, rec))
    print(f"threshold {t:.2f}   caught {tp:3d}  false alarms {fp:4d}  missed {fn:3d}   "
          f"precision {prec:.2f}  recall {rec:.2f}")

Now look hard at that table, because there's something important in it:

**The model did not change.**

One set of scores. Five completely different systems, with completely different
operational characteristics, from one trained model.

Moving the threshold moves you along a curve — and choosing *where on that curve
to sit* is a question about costs. It is not a machine learning question at all,
and it very often gets left at 0.5 by default, which is to say it gets decided by
whoever wrote the tutorial rather than by whoever owns the money.

Let's decide it properly instead.

In [ ]:
COST_FALSE_ALARM = 4        # an analyst spends five minutes
COST_MISSED_FRAUD = 500     # you eat the chargeback

best = None
for t in np.linspace(0.05, 0.95, 91):
    tp, fp, fn, tn = confusion(truth, (scores > t).astype(int))
    cost = fp * COST_FALSE_ALARM + fn * COST_MISSED_FRAUD
    if best is None or cost < best[1]:
        best = (t, cost, fp, fn)

print(f"cost-optimal threshold {best[0]:.2f}  ->  total cost {best[1]:,}")
print(f"  {best[2]} false alarms, {best[3]} missed frauds")
d = confusion(truth, (scores > 0.5).astype(int))
print(f"default 0.50 threshold  ->  total cost "
      f"{d[1] * COST_FALSE_ALARM + d[2] * COST_MISSED_FRAUD:,}")

Compare those two totals. Same model, same data, no retraining — a materially
better system, from changing one number.

Two numbers — what a false positive costs, and what a false negative costs —
turn an unanswerable question into arithmetic.

Getting those two numbers from whoever owns the problem is the
highest-leverage half hour in most machine learning projects.

It is also very often never done at all. People will spend six weeks on model
architecture and zero minutes on the number that turns out to matter more. Be the
person who asks.

In [ ]:
ths = np.linspace(0.01, 0.99, 200)
prec, rec = [], []
for t in ths:
    tp, fp, fn, tn = confusion(truth, (scores > t).astype(int))
    prec.append(tp / max(tp + fp, 1)); rec.append(tp / max(tp + fn, 1))

plt.figure(figsize=(5, 3.2))
plt.plot(rec, prec)
plt.axhline(truth.mean(), ls=":", c="grey")
plt.text(0.55, truth.mean() + 0.02, "random guessing", fontsize=8, color="grey")
plt.xlabel("recall"); plt.ylabel("precision"); plt.title("precision–recall curve")
plt.tight_layout()

Report **PR-AUC** for imbalanced problems and **ROC-AUC** for balanced ones, and
here's the reason, which is worth understanding rather than memorising.

ROC's x-axis is the false-positive *rate*. When 99% of your data is negative, a
great many false positives still make a small *rate* — so ROC looks flattering on
imbalanced problems, sometimes wildly so.

Precision has the raw count of false positives in its denominator, so it stays
honest. It can't be fooled by a large negative class.

And a single number is still a summary of a whole curve. Quote the curve, or at
absolute minimum quote precision and recall **at the threshold you actually
intend to use**.

## Calibration

In [ ]:
def calibration(probs, y, bins=8):
    edges = np.linspace(0, 1, bins + 1)
    out = []
    for lo, hi in zip(edges, edges[1:]):
        m = (probs >= lo) & (probs < hi)
        if m.sum() > 20:
            out.append((probs[m].mean(), y[m].mean(), int(m.sum())))
    return np.array(out)

cal = calibration(scores, truth)
plt.figure(figsize=(4.4, 3.4))
plt.plot([0, 1], [0, 1], "k:", label="perfectly calibrated")
plt.plot(cal[:, 0], cal[:, 1], "o-", label="this model")
plt.xlabel("predicted probability"); plt.ylabel("observed frequency")
plt.legend(); plt.tight_layout()
print("when it says 0.7, does it happen 70% of the time?")

That printed question is the whole concept. A model is **calibrated** if, among
the cases it scored 0.7, roughly 70% turn out positive.

And here's the subtlety worth having: **ranking correctly and being calibrated
are different properties.** A model can put every case in exactly the right order
and still have all its probabilities squashed into 0.4–0.6, or spread out to 0
and 1. For ranking, that's fine. For anything where the probability itself feeds
a decision — expected value, triage, pricing — it's not.

Neural networks are typically *over*confident. The usual fixes are Platt scaling
or temperature scaling on a held-out set, and both are about ten lines.

## After it ships

**Distribution shift.** A model learns the distribution it was shown. When the
world moves — a new competitor, a new product line, a pandemic — performance
decays and *nothing errors*.

Monitor the **inputs** as well as the outputs. Input drift is visible
immediately; label-based metrics might take months to arrive, and by then you've
been quietly wrong for a quarter.

**The feedback loop.** A fraud model that blocks transactions never finds out
whether those transactions were fraudulent. A recommender that shows five items
only ever gets feedback on five items.

Your training data becomes a product of your own past predictions — and this is a
genuinely hard problem, not a tidiness issue. The usual mitigation is to hold out
a small random slice of traffic that bypasses the model entirely, so you keep
seeing unfiltered reality.

**Training/serving skew.** The single most common production bug in the field:
features computed one way in the training pipeline and a subtly different way at
serving time. A mean imputed from the training set. A category encoded in a
different order. A timezone.

The defence is architectural, not disciplinary — **compute features with the same
code in both paths.** Don't rely on remembering.

**A model is a function, and you already know how to ship functions.** Version
it, pin its dependencies, log its inputs and outputs, health-check it, keep the
previous one warm.

Most of what gets called MLOps is ordinary operations applied to an artefact that
happens to be a matrix. You are not starting from zero here — you've been doing
this for years.

Log the model version alongside every single prediction, from day one.

The first time somebody asks "why did it decline this application in March?", you
will need to know which weights answered — and if you didn't log it, no amount of
later effort recovers it. The model has been retrained eleven times since.

This costs one column and saves an entire terrible week.

## Reading a paper

You now have enough to get the gist of most machine learning papers. Genuinely —
this is not encouragement, it's an assessment.

The trick is to not read them front to back.

**Pass 1, five minutes.** Title, abstract, figures, conclusion. Figure 1 is
almost always the architecture or the headline result, and it's usually the
densest information in the whole paper.

After this pass you should be able to say what problem they attacked and whether
the result is interesting *to you*. Most papers should stop here, and that's not
a failure — it's triage.

**Pass 2, twenty minutes.** The method section, skipping proofs. Ask four
questions, all of which you can now answer:

1. What is the **input and output**? (Chapter 3)
2. What is the **loss**? (Chapters 4–5)
3. What is the **architecture**, in terms of pieces you already know? (Chapters 8–13)
4. What is the **baseline**, and is the comparison fair? (Chapter 6)

**Pass 3, an afternoon.** Reproduce something. This is where actual understanding
happens, and it's the pass that almost everybody skips.

Question 4 is the one that catches most weak papers, and you can ask it without
understanding a single equation.

Look for: a baseline tuned less carefully than the proposed method. A test set
that was clearly consulted repeatedly. A comparison against an old version of a
competitor. Results averaged over one seed.

None of those require mathematics to spot. All of them are common. And noticing
them is most of what "reading critically" actually means in this field.

Some notation that used to be opaque and now isn't:

<div class="table-scroll">

| Symbol | Reads as |
|---|---|
| $\theta$ | all the model's parameters, at once |
| $\mathcal{L}(\theta)$ | the loss, as a function of them |
| $\nabla_\theta$ | the gradient with respect to them |
| $\mathbb{E}_{x \sim D}[\,\cdot\,]$ | average over data drawn from distribution $D$ |
| $\arg\max_\theta$ | *the* $\theta$ that maximises this, not the value |
| $p_\theta(y \mid x)$ | probability the model gives $y$ given $x$ |
| $\hat{y}$ | prediction, as opposed to $y$, the truth |
| $\|\cdot\|_2$, $\|\cdot\|_1$ | Euclidean and absolute-value norms |
| $\odot$ | elementwise multiplication |

</div>

So this sentence:

$$\min_\theta \mathbb{E}_{(x,y)\sim D}\left[\mathcal{L}(f_\theta(x), y)\right] + \lambda\|\theta\|_2^2$$

says: *make the average loss on your data small, and penalise large weights while
you're at it.*

Which is the whole of Chapters 5 and 6, written in nine symbols. It looked like a
foreign language a fortnight ago. It's a sentence about a hill and a penalty.

## A checklist for your first real project

1. **Frame it.** Supervision, task, features, target. Write it down.
2. **Build the validation set first**, before any modelling, structured the way
   deployment will be.
3. **Compute a trivial baseline.** Majority class, or last week's value.
4. **Get the two costs** — false positive, false negative — from whoever owns the
   problem.
5. **Fit the boring model.** Logistic regression or gradient boosting.
6. **Look at the errors.** Not the metric — the actual rows it got wrong. This is
   the highest-value hour in any project and the one most often skipped.
7. **Only now** consider something more complicated.
8. **Check for leakage** whenever a result is better than you expected.
9. **Pick the threshold with the costs from step 4.**
10. **Ship it with logging**, and watch the input distribution.

Print it, put it somewhere. Steps 2, 4 and 6 are the ones people skip, and
they're the ones that decide whether the project works.

## Where to go next

**Consolidate.** Do a Kaggle playground competition end to end — not to win, but
to feel the loop of *validate, change one thing, measure*. A week of that is
worth a month of reading, and it's the fastest way to build the instinct.

**Go deeper on the code.** Karpathy's *Zero to Hero* series builds a GPT from
nothing and is the natural sequel to chapters 9 and 13. The fastai course is
excellent and takes the *opposite* route to this one — top down, library first —
which makes the two genuinely complementary rather than redundant. You'll get
more from it now than you would have a fortnight ago.

**Go deeper on the theory.** Bishop's *Pattern Recognition and Machine Learning*
for the probabilistic view; Goodfellow, Bengio and Courville's *Deep Learning*
for the standard reference. Both are demanding, and both are much better after
this than before it.

**Read papers.** *Attention Is All You Need* (2017) is the one to start with —
you've implemented most of it. Then simply follow whatever is cited by the thing
you actually want to build. That's how everybody does it; there's no reading list.

The single most useful habit, and the thing I'd most like you to keep:

**When you meet something new, find where it sits on [the map](/map/).**

Is it a new supervision signal? A new model family? A new piece of the fitting
machinery? Almost everything is one of those three.

Placing it correctly turns a strange new thing into a variation on something
you've already built — which is the difference between a field that feels
infinite and one that feels navigable.

Pick a problem from your own work. A real one — something you have data for, or
could get data for.

Write half a page:

1. Supervision, task, features, target.
2. How you'd build the validation set, and *why that structure*.
3. The trivial baseline, and what it would score.
4. The cost of a false positive and of a false negative.
5. The first model you'd try, and why.
6. One way this could be leakage, and how you'd check.
7. **What would make you abandon the project.**

If you can answer all seven, you can start.

If you can't answer 2, 4 or 7 — that's where the work is. And notice that none of
it is modelling work.

Every one of the first six questions has a technical answer you could look up.

Question 7 doesn't. And it's the one that decides whether the project is worth
doing at all.

Good abandonment criteria are **concrete and set in advance**. For example:

> *If a well-tuned gradient-boosted model cannot beat the baseline by 5 points on
> the validation set within two weeks, the signal isn't there.*

Or:

> *If fixing the label noise requires more annotator time than the project saves,
> stop.*

Written down beforehand, that is a **decision** — clean, defensible, and made
while everyone is still calm.

Discovered afterwards, it's four months and an awkward meeting.

The habit of writing question 7 down before you start is worth more than any
architecture in this book. It's also the one nobody teaches, because it isn't
technical and it doesn't demo well.

---

## That's the fortnight

You've built a linear model, a gradient descent optimiser, a neural network, a
backward pass, an attention mechanism, an embedding, an autoencoder and a
tokeniser — every one of them out of arrays and arithmetic,
with nothing hidden.

You know which of them to reach for. You know when the answer is none of them.
You can open a paper and place it on a map.

And I hope somewhere in there — the hinges adding up into a curve, the templates
turning out to be pictures of digits, the gradient walking downhill in the fog,
one triangle of booleans separating BERT from GPT — some of it was actually
*fun*. That was the other half of the point.

The field will keep moving. The frame won't.

<div class="pip-note">
  
  <p>Go and build something. That's the whole reason for any of this.</p>
</div>

If you want more, the [extras](/extras/) are everything that didn't fit in
fourteen days — the roads not taken, the deeper whys, and the things you only
learn by shipping.